# Resource estimation for SOSSA sum-of-squares spectral amplification

This notebook estimates the cost of running unary-iteration quantum phase estimation on a
Hamiltonian block-encoded with **SOSSA** (sum-of-squares spectral amplification, Low *et al.*,
[arXiv:2502.15882](https://arxiv.org/abs/2502.15882)).

We demonstrate the resource estimation in three workflows:
1. [Part 1 --- stored DFTHC H<sub>2</sub>](#part-1) loads a factorized Hamiltonian from JSON.
2. [Part 2 --- stretched N<sub>2</sub>](#part-2) factorizes the Hamiltonian using the double factorization (DF).
3. [Part 3 --- synthetic examples](#part-3) provides resource estimates based on $(N,R,B,C)$ shapes.

Each part produces a unary-iteration QPE circuit and resource estimates from `qdk.qre`.

In addition to [installing `qdk-chemistry`](https://github.com/microsoft/qdk-chemistry/blob/main/INSTALL.md),
you will need the `qre`, `jupyter` and `qiskit-extras` extras:

```bash
pip install 'qdk-chemistry[jupyter,qiskit-extras,qre]'
```

In [ ]:
import math
from pathlib import Path

import pandas as pd

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    Hamiltonian,
    MajoranaMapping,
    StateVectorContainer,
    Wavefunction,
)
from qdk_chemistry.utils import Logger


TARGET_PRECISION = 1e-3  # Hartree ("chemical accuracy" is ~1.6e-3)
MAX_ERROR = 0.01

# Reduce logging output for the demo
Logger.set_global_level(Logger.LogLevel.off)

## QPE Circuit Builder

The QPE Circuit Builder from an input factorized hamiltonian is the same.
`SOSSAQubitMapper` turns a `FactorizedHamiltonianContainer` into a structured qubit operator that
records the generator coefficients and scalar shift $E_{\mathrm{SOS}}$. `SOSSABuilder` derives the
generator one-norms and block-encoding normalization $\Lambda$, then the unary-iteration QPE builder
consumes the operator directly: it allocates one phase qubit per query and reuses a single controlled
walk across every slot.

The circuit-mapper settings choose how each piece of the walk is synthesized:

- `outer_prepare` --- prepares $\sqrt{c_x}$-weighted amplitudes over the generator index.
- `inner_prepare_algorithm` --- the controlled rotation cascade that builds each Majorana bilinear.
- `select_algorithm` --- `qrom_phase_gradient` applies the basis rotations through a QROM lookup
  into a shared phase-gradient register, which trades rotation synthesis for Toffolis.
- `rotation_bit_precision` / `coefficient_bit_precision` --- how finely the Givens angles and the
  amplitudes are discretized.

In [ ]:
def sossa_unary_qpe_circuit(
    hamiltonian,
    *,
    num_queries,
    n_alpha,
    n_beta,
    circuit_mapper=None,
    rotation_bit_precision=15,
    coefficient_bit_precision=11,
):
    """Build a unary-iteration QPE circuit driven by the SOSSA walk.

    Args:
        hamiltonian: A Hamiltonian backed by a FactorizedHamiltonianContainer.
        num_queries: Number of walk applications in the phase-estimation schedule.
        n_alpha: Number of alpha electrons in the reference determinant.
        n_beta: Number of beta electrons in the reference determinant.
        circuit_mapper: Optional SOSSA circuit mapper; defaults to the resource-estimation configuration.
        rotation_bit_precision: Bits used to discretize the Givens rotation angles.
        coefficient_bit_precision: Bits used to discretize the PREPARE amplitudes.

    Returns:
        A tuple of (QPE circuit, SOSSA qubit operator).

    """
    container = hamiltonian.get_container()
    num_orbitals = container.get_num_orbitals()
    orbitals = container.get_orbitals()

    # `SOSSAQubitMapper` turns a factorized Hamiltonian container into a structured qubit operator 
    operator = create("qubit_mapper", "sossa").run(
        hamiltonian, MajoranaMapping.jordan_wigner(2 * num_orbitals)
    )

    # Hartree-Fock reference determinant, loaded with the sparse isometry method.
    hf_config = Configuration.canonical_hf_configuration(n_alpha, n_beta, num_orbitals)
    reference = Wavefunction(StateVectorContainer(hf_config, orbitals))
    state_prep = create("state_prep", "sparse_isometry").run(reference)

    if circuit_mapper is None:
        circuit_mapper = AlgorithmRef(
            "circuit_mapper",
            "sossa",
            outer_prepare=AlgorithmRef("state_prep", "alias_sampling"),
            inner_prepare_algorithm="controlled_alias_sampling",
            select_algorithm="qrom_phase_gradient",
            rotation_bit_precision=rotation_bit_precision,
            coefficient_bit_precision=coefficient_bit_precision,
        )

    builder = create(
        "qpe_circuit_builder",
        "qdk_unary",
        num_queries=num_queries,
        circuit_mapper=circuit_mapper,
        unitary_builder=AlgorithmRef("hamiltonian_unitary_builder", "sossa"),
    )
    circuit = builder.run(state_preparation=state_prep, qubit_hamiltonian=operator)[0]
    return circuit, operator


def heisenberg_queries(lambda_sos, target_precision):
    """Queries needed to resolve ``target_precision`` at the amplified band edge.

    The unary schedule wants ``num_queries + 1`` to be a power of two, so round up.
    """
    ideal = math.pi * lambda_sos / (2.0 * target_precision)
    return 2 ** math.ceil(math.log2(max(ideal, 2.0))) - 1

<a id="part-1"></a>

## Part 1 --- a stored DFTHC Hamiltonian

The first instance ships with the repository as a serialized `FactorizedHamiltonianContainer`.
It is a minimal H<sub>2</sub> problem, $N = 2$ spatial orbitals with $R = 1$ rank, $B = 2$ bases
and $C = 1$ copy.

In [ ]:
json_path = Path("data") / "h2_factorized_r1_b2_c1.hamiltonian.json"
h2_hamiltonian = Hamiltonian.from_json(json_path.read_text())
h2_container = h2_hamiltonian.get_container()

print(h2_hamiltonian.get_summary())

### Mapping to the SOSSA qubit operator

The mapper reads the factorization and emits the generator list together with $E_{\mathrm{SOS}}$, the
scalar that converts an $H_{\mathrm{gap}}$ eigenvalue back into a physical energy. The unitary builder
then derives $\Lambda$, half the sum of squared generator one-norms, which sets the block-encoding
normalization.

In [ ]:
h2_operator = create("qubit_mapper", "sossa").run(
    h2_hamiltonian, MajoranaMapping.jordan_wigner(2 * h2_container.get_num_orbitals())
)
h2_metadata = create("hamiltonian_unitary_builder", "sossa").run(h2_operator).get_container().metadata
h2_queries = heisenberg_queries(h2_metadata.normalization, TARGET_PRECISION)

print(f"Block-encoding normalization  Lambda = {h2_metadata.normalization:.6f} Hartree")
print(f"Sum-of-squares shift         E_SOS  = {h2_metadata.energy_shift:.6f} Hartree")
print(f"System qubits                       = {h2_operator.get_container().num_qubits}")
print(f"Queries for {TARGET_PRECISION:.0e} Ha at Lambda = {h2_metadata.normalization:.4f}: {h2_queries}")

### Choosing the query schedule and sampling the circuit

Unary-iteration QPE applies the walk `num_queries` times and reads the phase off a register of
the same width. To resolve an energy $\sigma_E$ at the amplified band edge we need roughly
$\pi \Lambda / (2\sigma_E)$ queries; we round up to the next power of two minus one so the
schedule divides evenly.

The full-precision circuit is intended for resource estimation rather than local simulation. Before
estimating it, we sample a compact 7-query circuit using exact dense/direct synthesis, which avoids
the large alias-sampling and phase-gradient work registers while exercising the same SOSSA walk.

In [ ]:
# This cell takes ~30 seconds to run
from qdk.widgets import Histogram

SIMULATION_QUERIES = 7
SIMULATION_SHOTS = 256
simulation_mapper = AlgorithmRef(
    "circuit_mapper",
    "sossa",
    outer_prepare=AlgorithmRef("state_prep", "dense_pure_state"),
    inner_prepare_algorithm="direct",
    select_algorithm="direct",
)
h2_circuit, _ = sossa_unary_qpe_circuit(
    h2_hamiltonian,
    num_queries=SIMULATION_QUERIES,
    n_alpha=1,
    n_beta=1,
    circuit_mapper=simulation_mapper,
)
h2_execution = create("circuit_executor", "qdk_sparse_state_simulator", seed=20250815).run(
    h2_circuit,
    shots=SIMULATION_SHOTS,
)
h2_phase_probabilities = {
    bitstring: count / h2_execution.total_shots
    for bitstring, count in h2_execution.bitstring_counts.items()
}
display(Histogram(bar_values=h2_phase_probabilities))

### Physical resource estimates

`qdk.qre` maps the logical circuit onto a fault-tolerant architecture and returns the
Pareto-optimal trade-offs between physical qubit count and runtime. We use a Majorana-based
architecture at a $10^{-5}$ physical error rate with the `ThreeAux` code and round-based magic
state factories, and budget 1% total error.

In [ ]:
from qdk.qre import estimate, plot_estimates
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

architecture = Majorana(error_rate=1e-5)
isa_query = ThreeAux.q() * RoundBasedFactory.q(use_cache=True, code_query=ThreeAux.q())

h2_results = estimate(
    h2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_H2"
)
h2_results.add_factory_summary_column()
display(h2_results.as_frame())

plot_estimates(h2_results, figsize=(6, 4), runtime_unit="ms")

<a id="part-2"></a>

## Part 2 --- stretched N<sub>2</sub> from a structure file

The classical preparation is condensed here to keep the focus on SOSSA resource estimation. See the
[stretched N<sub>2</sub> QPE notebook](qpe_stretched_n2.ipynb) for the full workflow and discussion.

In [ ]:
from qdk_chemistry.data import Structure
from qdk_chemistry.data.symmetry import SymmetryLabel, axes
from qdk_chemistry.utils import compute_valence_space_parameters

structure = Structure.from_xyz_file(Path("data/stretched_n2.structure.xyz"))
scf_solver = create("scf_solver")
e_hf, wfn_hf = scf_solver.run(
    structure,
    charge=0,
    spin_multiplicity=1,
    basis_or_guess="cc-pvdz",
)

num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
active_space_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o,
)
valence_wfn = active_space_selector.run(wfn_hf)

localizer = create("orbital_localizer", "qdk_mp2_natural_orbitals")
valence_indices = valence_wfn.get_orbitals().active_indices()
localized_wfn = localizer.run(
    valence_wfn,
    list(valence_indices.indices(SymmetryLabel([axes.alpha()]))),
    list(valence_indices.indices(SymmetryLabel([axes.beta()]))),
)

hamiltonian_constructor = create("hamiltonian_constructor")
localized_hamiltonian = hamiltonian_constructor.run(localized_wfn.get_orbitals())
num_alpha_electrons, num_beta_electrons = localized_wfn.get_active_num_electrons()
selected_ci = create(
    "multi_configuration_calculator",
    "macis_asci",
    calculate_one_rdm=True,
    calculate_two_rdm=True,
)
_, selected_ci_wfn = selected_ci.run(localized_hamiltonian, num_alpha_electrons, num_beta_electrons)

autocas = create("active_space_selector", "qdk_autocas_eos")
autocas_wfn = autocas.run(selected_ci_wfn)
active_hamiltonian = hamiltonian_constructor.run(autocas_wfn.get_orbitals())
alpha_electrons, beta_electrons = autocas_wfn.get_active_num_electrons()

casci = create("multi_configuration_calculator", "macis_cas")
e_cas, _ = casci.run(active_hamiltonian, alpha_electrons, beta_electrons)
print(f"Hartree-Fock energy: {e_hf:.6f} Hartree")
print(f"Active-space CASCI energy: {e_cas:.6f} Hartree")
print(f"Active space: {alpha_electrons} alpha + {beta_electrons} beta electrons")

### Double Factorization

In [ ]:
factorizer = create("double_factorizer", "eigen_decomposition")
factorizer.settings().set("truncation_threshold", 1e-8)
n2_hamiltonian = factorizer.run(active_hamiltonian)
n2_container = n2_hamiltonian.get_container()

print(n2_hamiltonian.get_summary())

### Physical resource estimates

In [ ]:
n2_operator = create("qubit_mapper", "sossa").run(
    n2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n2_container.get_num_orbitals())
)
n2_metadata = create("hamiltonian_unitary_builder", "sossa").run(n2_operator).get_container().metadata

print(f"Block-encoding normalization  Lambda = {n2_metadata.normalization:.6f} Hartree")
print(f"Sum-of-squares shift         E_SOS  = {n2_metadata.energy_shift:.6f} Hartree")
print(f"System qubits                       = {n2_operator.get_container().num_qubits}")

n2_queries = heisenberg_queries(n2_metadata.normalization, TARGET_PRECISION)
print(f"Heisenberg-limited queries for {TARGET_PRECISION:.0e} Ha: {n2_queries}")

n2_circuit, _ = sossa_unary_qpe_circuit(
    n2_hamiltonian,
    num_queries=n2_queries,
    n_alpha=alpha_electrons,
    n_beta=beta_electrons,
)
n2_results = estimate(
    n2_circuit.get_qre_application(), architecture, isa_query, max_error=0.01, name="SOSSA_N2"
)
n2_results.add_factory_summary_column()
display(n2_results.as_frame())

plot_estimates(n2_results, figsize=(6, 4), runtime_unit="ms")

<a id="part-3"></a>

## Part 3 --- molecular-scale synthetic examples

This section repeats the SOSSA resource-estimation procedure for four molecular active spaces. Each
example specifies the factorized-Hamiltonian dimensions

- $N$: number of spatial orbitals,
- $R$: number of factorization ranks,
- $B$: number of basis vectors per rank, and
- $C$: number of coefficient copies per basis vector.

It also specifies the effective normalization $\lambda_{\mathrm{eff}}$ and the rotation and coefficient
precisions $b_{\mathrm{rot}}$ and $b_{\mathrm{coeff}}$. For target precision $\sigma_E$.

The tensors are deterministic synthetic data with the requested $(N,R,B,C)$ dimensions, so the estimates are approximations.

In [ ]:
import numpy as np
from qdk_chemistry.data import FactorizedHamiltonianContainer, ModelOrbitals

MOLECULES = {
    "Fe2S2 (30e, 20o)": {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    },
    "FeMoCo (54e, 54o)": {
        "electrons": 54,
        "N": 54,
        "R": 10,
        "B": 27,
        "C": 27,
        "lambda_eff": 21.3674,
        "b_coeff": 9,
        "b_rot": 16,
    },
    "CPD1-P450X (63e, 58o)": {
        "electrons": 63,
        "N": 58,
        "R": 9,
        "B": 29,
        "C": 14,
        "lambda_eff": 32.7923,
        "b_coeff": 10,
        "b_rot": 15,
    },
    "CO2-XVIII (64e, 56o)": {
        "electrons": 64,
        "N": 56,
        "R": 5,
        "B": 28,
        "C": 28,
        "lambda_eff": 17.0712,
        "b_coeff": 7,
        "b_rot": 12,
    },
}


def make_fake_hamiltonian(n: int, r: int, b: int, c: int, seed: int = 42) -> Hamiltonian:
    """Create a deterministic synthetic factorized Hamiltonian.

    Args:
        n: Number of spatial orbitals.
        r: Number of factorization ranks.
        b: Number of basis vectors per rank.
        c: Number of coefficient copies per basis vector.
        seed: Random seed used to generate the synthetic tensors.

    Returns:
        A Hamiltonian backed by a synthetic factorized container with dimensions `(N, R, B, C)`.
    """
    rng = np.random.default_rng(seed)
    one_body = rng.standard_normal((n, n))
    one_body = (one_body + one_body.T) / 2

    basis_vectors = rng.standard_normal((r, b, n))
    basis_vectors /= np.linalg.norm(basis_vectors, axis=-1, keepdims=True)

    container = FactorizedHamiltonianContainer(
        0.0,
        basis_vectors.ravel(),
        rng.standard_normal((r, b, c)).ravel() * 0.1,
        rng.standard_normal((r, c)) * 0.1,
        one_body,
        np.zeros_like(one_body),
        ModelOrbitals(n),
    )
    return Hamiltonian(container)

### Physical resource estimates
This cell runs one baseline and two fixed memory/compute sweeps per molecule.

In [ ]:
# This cell takes 2 mins to run.

from qdk.qre import DynamicMemoryCompute, LatticeSurgery, PSSPC, estimate
from qdk.qre.instruction_ids import T
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

MEMORY_COMPUTE_MODES = [10, 30]

architecture = Majorana(error_rate=1e-5)
psspc_query = PSSPC.q(num_ts_per_rotation=list(range(28, 43)))
lattice_surgery_query = LatticeSurgery.q(
    slow_down_factor=[float(value) for value in range(1, 35)]
)
trace_queries = {"No memory/compute": psspc_query * lattice_surgery_query}
trace_queries.update(
    {
        f"{compute_percentage}% compute": DynamicMemoryCompute.q(
            compute_capacity_percentage=compute_percentage / 100.0
        )
        * psspc_query
        * lattice_surgery_query
        for compute_percentage in MEMORY_COMPUTE_MODES
    }
)

logical_rows = []
physical_frames = []

for molecule, params in MOLECULES.items():
    n = int(params["N"])
    r = int(params["R"])
    b = int(params["B"])
    c = int(params["C"])
    num_electrons = int(params["electrons"])
    lambda_eff = float(params["lambda_eff"])
    b_coeff = int(params["b_coeff"])
    b_rot = int(params["b_rot"])
    num_queries = heisenberg_queries(lambda_eff, TARGET_PRECISION)

    fake_hamiltonian = make_fake_hamiltonian(n, r, b, c)
    circuit, _ = sossa_unary_qpe_circuit(
        fake_hamiltonian,
        num_queries=num_queries,
        n_alpha=(num_electrons + 1) // 2,
        n_beta=num_electrons // 2,
        rotation_bit_precision=b_rot,
        coefficient_bit_precision=b_coeff,
    )

    logical_counts = dict(circuit.estimate().logical_counts)
    logical_rows.append(
        {
            "Molecule": molecule,
            "N": n,
            "R": r,
            "B": b,
            "C": c,
            "Electrons": num_electrons,
            "lambda_eff (Ha)": lambda_eff,
            "b_coeff": b_coeff,
            "b_rot": b_rot,
            "Queries": num_queries,
            "Logical qubits": int(logical_counts["numQubits"]),
            "Logical Toffolis": int(
                logical_counts["cczCount"] + logical_counts.get("ccixCount", 0)
            ),
        }
    )

    for memory_mode, trace_query in trace_queries.items():
        isa_query = ThreeAux.q(
            distance=list(range(11, 42, 2))
        ) * RoundBasedFactory.q(
            use_cache=True,
            code_query=ThreeAux.q(distance=list(range(5, 42, 2))),
        )
        result = estimate(
            circuit.get_qre_application(),
            architecture,
            isa_query,
            trace_query,
            max_error=MAX_ERROR,
            name=f"{molecule} ({memory_mode})",
        )
        result.add_qubit_partition_column()
        result.add_column(
            "num_magic_states", lambda entry: entry.factories[T].states
        )
        result.add_factory_summary_column()

        physical_frame = result.as_frame().copy()
        physical_frame.insert(0, "Molecule", molecule)
        physical_frame.insert(1, "Memory/compute", memory_mode)
        physical_frame.insert(2, "Pareto point", range(1, len(physical_frame) + 1))
        physical_frames.append(physical_frame.drop(columns="name", errors="ignore"))


part3_logical_estimates = pd.DataFrame(logical_rows).set_index("Molecule")
part3_physical_estimates = pd.concat(physical_frames, ignore_index=True).set_index(
    ["Molecule", "Memory/compute", "Pareto point"]
)

display(part3_logical_estimates.style.format({"Logical Toffolis": "{:.3e}"}))
display(
    part3_physical_estimates.style.format(
        {
            "qubits": "{:,}",
            "physical_compute_qubits": "{:,}",
            "physical_factory_qubits": "{:,}",
            "physical_memory_qubits": "{:,}",
            "num_magic_states": "{:.3e}",
            "runtime": lambda value: f"{value.total_seconds() / 3600:.2f} h",
        }
    )
)